# Data Cleaning

## Objective
This notebook cleans the contaminated retail datasets by handling missing values, duplicates, incorrect data types, inconsistent values, and validating data quality before analysis and modeling.

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
sales = pd.read_csv("../data/raw/retail_contaminated_dataset/sales_transactions.csv")
customers = pd.read_csv("../data/raw/retail_contaminated_dataset/customer_master.csv")
inventory = pd.read_csv("../data/raw/retail_contaminated_dataset/inventory_snapshot.csv")
sku = pd.read_csv("../data/raw/retail_contaminated_dataset/sku_master.csv")
stores = pd.read_csv("../data/raw/retail_contaminated_dataset/store_master.csv")
promotions = pd.read_csv("../data/raw/retail_contaminated_dataset/promotions.csv")
flags = pd.read_csv("../data/raw/retail_contaminated_dataset/sku_inventory_flags.csv")

## Customer Master Data Cleaning

In [4]:
# Preview the first 5 rows of the customer dataset
customers.head()

,cust_id,age,gender,city,loyalty_segment,preferred_channel,registration_date
0,CUST00001,52,Female,Sukkur,Silver,Online,2021-01-01
1,CUST00002,41,Female,Karachi,Silver,Mobile App,2021-05-14
2,CUST00003,48,Female,Bahawalpur,Bronze,In-Store,2022-10-23
3,CUST00004,45,Male,Karachi,Silver,In-Store,2015-07-15
4,CUST00005,39,Female,Faisalabad,Bronze,In-Store,2024-01-02


In [5]:
# Check the number of rows and columns
customers.shape

(10000, 7)

In [6]:
# Display dataset structure, data types, and non-null counts
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   cust_id            10000 non-null  object
 1   age                10000 non-null  int64 
 2   gender             10000 non-null  object
 3   city               10000 non-null  object
 4   loyalty_segment    10000 non-null  object
 5   preferred_channel  10000 non-null  object
 6   registration_date  10000 non-null  object
dtypes: int64(1), object(6)
memory usage: 547.0+ KB


In [7]:
# Check for missing values in each column
customers.isnull().sum()

cust_id              0
age                  0
gender               0
city                 0
loyalty_segment      0
preferred_channel    0
registration_date    0
dtype: int64

In [8]:
# Check for duplicate records
customers.duplicated().sum()

np.int64(0)

In [9]:
# Generate descriptive statistics
customers.describe(include="all")

,cust_id,age,gender,city,loyalty_segment,preferred_channel,registration_date
count,10000,10000.000000,10000,10000,10000,10000,10000
unique,10000,NaN,3,15,4,3,3661
top,CUST00001,NaN,Female,Karachi,Bronze,In-Store,2025-02-02
freq,1,NaN,4930,1577,3969,5512,10
mean,NaN,37.702900,NaN,NaN,NaN,NaN,NaN
std,NaN,12.190106,NaN,NaN,NaN,NaN,NaN
min,NaN,18.000000,NaN,NaN,NaN,NaN,NaN
25%,NaN,29.000000,NaN,NaN,NaN,NaN,NaN
50%,NaN,37.000000,NaN,NaN,NaN,NaN,NaN
75%,NaN,46.000000,NaN,NaN,NaN,NaN,NaN


In [10]:
# Count unique values in each column
customers.nunique()

cust_id              10000
age                     63
gender                   3
city                    15
loyalty_segment          4
preferred_channel        3
registration_date     3661
dtype: int64

In [11]:
# Convert registration_date to datetime
customers["registration_date"] = pd.to_datetime(
    customers["registration_date"],
    errors="coerce"
)

customers["registration_date"].dtype

dtype('<M8[ns]')

In [12]:
# Check if any dates became missing after conversion
customers["registration_date"].isnull().sum()

np.int64(0)

In [13]:
# Display summary statistics for age
customers["age"].describe()

count    10000.000000
mean        37.702900
std         12.190106
min         18.000000
25%         29.000000
50%         37.000000
75%         46.000000
max         80.000000
Name: age, dtype: float64

In [14]:
# Identify unrealistic age values
customers[(customers["age"] < 18) | (customers["age"] > 80)]

,cust_id,age,gender,city,loyalty_segment,preferred_channel,registration_date


In [15]:
# Display unique gender values
customers["gender"].value_counts(dropna=False)

gender
Female    4930
Male      4885
Other      185
Name: count, dtype: int64

In [16]:
# Display loyalty segment categories
customers["loyalty_segment"].value_counts()

loyalty_segment
Bronze      3969
Silver      3106
Gold        1905
Platinum    1020
Name: count, dtype: int64

In [17]:
# Display preferred shopping channels
customers["preferred_channel"].value_counts()

preferred_channel
In-Store      5512
Online        2994
Mobile App    1494
Name: count, dtype: int64

In [18]:
# Display city names
customers["city"].value_counts()

city
Karachi       1577
Lahore        1418
Islamabad      996
Rawalpindi     832
Faisalabad     796
Peshawar       640
Multan         602
Sialkot        470
Gujranwala     451
Quetta         449
Sukkur         390
Hyderabad      374
Abbottabad     354
Bahawalpur     342
Sargodha       309
Name: count, dtype: int64

In [19]:
# Save cleaned customer dataset
customers.to_csv(
    "../data/processed/customer_master_clean.csv",
    index=False
)

print("Customer dataset saved successfully.")

Customer dataset saved successfully.


## Customer Master Cleaning Summary

Cleaning Results

- Converted `registration_date` to datetime format.
- No missing values found.
- No duplicate records found.
- All customer ages fall within the expected range (18–80 years).
- Gender values are consistent.
- Loyalty segments are standardized.
- Preferred shopping channels are consistent.
- City names contain no inconsistencies.

**Conclusion:** No additional cleaning was required for the Customer Master dataset.

## Sales Transactions Data Cleaning

The sales transactions dataset records every customer purchase made across retail stores. This section inspects data quality issues and performs necessary cleaning before the dataset is used for analysis and modeling.

In [20]:
# Preview the first 5 rows of the sales dataset
sales.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,NaN
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,NaN
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,NaN
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,NaN
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,NaN


In [21]:
# Check the number of rows and columns
sales.shape

(9945511, 11)

In [22]:
# Display dataset structure, data types, and non-null counts
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9945511 entries, 0 to 9945510
Data columns (total 11 columns):
 #   Column        Dtype  
---  ------        -----  
 0   date          object 
 1   receipt_id    object 
 2   store_id      object 
 3   sku_id        object 
 4   customer_id   object 
 5   quantity      int64  
 6   unit_price    float64
 7   total_value   float64
 8   channel       object 
 9   discount_pct  float64
 10  promo_id      object 
dtypes: float64(3), int64(1), object(7)
memory usage: 834.7+ MB


In [23]:
# Check for missing values in each column
sales.isnull().sum()

date                  0
receipt_id            0
store_id              0
sku_id                0
customer_id           0
quantity              0
unit_price            0
total_value           0
channel               0
discount_pct          0
promo_id        7855345
dtype: int64

In [24]:
# Check for duplicate records
sales.duplicated().sum()

np.int64(12897)

In [25]:
# Generate descriptive statistics
sales.describe(include="all")

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
count,9945511,9945511,9945511,9945511,9945511,9.945511e+06,9.945511e+06,9.945511e+06,9945511,9.945511e+06,2090166
unique,1461,5167864,30,5000,10000,NaN,NaN,NaN,3,NaN,96
top,2025-12-29,RCPT01092062,ST29,SKU04321,CUST06744,NaN,NaN,NaN,In-Store,NaN,PROMO045
freq,9856,5,472578,630147,2303,NaN,NaN,NaN,5496812,NaN,185809
mean,NaN,NaN,NaN,NaN,NaN,1.880336e+00,6.215749e+02,1.094512e+03,NaN,6.289965e+00,NaN
std,NaN,NaN,NaN,NaN,NaN,1.089063e+00,6.751952e+02,1.536814e+03,NaN,1.383267e+01,NaN
min,NaN,NaN,NaN,NaN,NaN,1.000000e+00,2.441000e+01,1.225000e+01,NaN,0.000000e+00,NaN
25%,NaN,NaN,NaN,NaN,NaN,1.000000e+00,1.734300e+02,2.351700e+02,NaN,0.000000e+00,NaN
50%,NaN,NaN,NaN,NaN,NaN,2.000000e+00,3.367200e+02,5.432000e+02,NaN,0.000000e+00,NaN
75%,NaN,NaN,NaN,NaN,NaN,3.000000e+00,9.256700e+02,1.277610e+03,NaN,0.000000e+00,NaN


In [26]:
# Count unique values in each column
sales.nunique()

date               1461
receipt_id      5167864
store_id             30
sku_id             5000
customer_id       10000
quantity              5
unit_price         4837
total_value      182745
channel               3
discount_pct         84
promo_id             96
dtype: int64

In [27]:
# Display unique values for categorical columns
for col in sales.select_dtypes(include="object").columns:
    print(f"\n{col}")
    print(sales[col].value_counts())


date
date
2025-12-29    9856
2025-12-22    9796
2024-12-25    9736
2025-12-30    9706
2025-12-31    9679
              ... 
2022-01-26    5021
2022-01-06    5014
2022-01-05    4972
2022-01-28    4941
2022-01-09    4880
Name: count, Length: 1461, dtype: int64

receipt_id
receipt_id
RCPT01092062    5
RCPT02905094    5
RCPT01658810    5
RCPT02904919    5
RCPT01658792    5
               ..
RCPT01366472    1
RCPT01366471    1
RCPT01366470    1
RCPT03306245    1
RCPT05181384    1
Name: count, Length: 5167864, dtype: int64

store_id
store_id
ST29    472578
ST06    472418
ST20    471940
ST07    471603
ST30    469839
ST22    469715
ST21    469288
ST28    354827
ST18    354667
ST05    354467
ST17    354338
ST27    354267
ST14    354257
ST08    353504
ST11    353331
ST09    353255
ST04    353138
ST03    352756
ST26    237087
ST01    236295
ST10    236204
ST24    236190
ST16    236126
ST13    236065
ST19    235387
ST25    235170
ST15    235122
ST12    234974
ST23    234745
ST02    161958
Name: c

In [28]:
# Display summary statistics for numeric columns
sales.describe()

,quantity,unit_price,total_value,discount_pct
count,9.945511e+06,9.945511e+06,9.945511e+06,9.945511e+06
mean,1.880336e+00,6.215749e+02,1.094512e+03,6.289965e+00
std,1.089063e+00,6.751952e+02,1.536814e+03,1.383267e+01
min,1.000000e+00,2.441000e+01,1.225000e+01,0.000000e+00
25%,1.000000e+00,1.734300e+02,2.351700e+02,0.000000e+00
50%,2.000000e+00,3.367200e+02,5.432000e+02,0.000000e+00
75%,3.000000e+00,9.256700e+02,1.277610e+03,0.000000e+00
max,5.000000e+00,4.755470e+03,2.377735e+04,4.980000e+01


In [29]:
# Display the dataset
sales

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,NaN
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,NaN
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,NaN
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,NaN
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
9945506,2023-09-09,RCPT05181381,ST04,SKU00701,CUST06397,2,39.24,68.51,In-Store,12.7,PROMO010
9945507,2025-12-03,RCPT05181382,ST03,SKU02163,CUST04945,1,791.70,791.70,In-Store,0.0,NaN
9945508,2025-12-03,RCPT05181382,ST03,SKU02866,CUST04945,4,94.26,377.04,In-Store,0.0,NaN
9945509,2023-01-20,RCPT05181383,ST03,SKU03727,CUST05244,1,1660.93,1660.93,In-Store,0.0,NaN


In [30]:
# Convert date column to datetime
sales["date"] = pd.to_datetime(
    sales["date"],
    errors="coerce"
)

sales["date"].dtype

dtype('<M8[ns]')

In [31]:
# Check if any dates became missing after conversion
sales["date"].isnull().sum()

np.int64(0)

In [32]:
# Check duplicate transactions
sales.duplicated().sum()

np.int64(12897)

In [33]:
# Display duplicate rows
sales[sales.duplicated()].head(10)

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
272,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
315,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
545,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
2936,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
5846,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN
6711,2022-07-13,RCPT00003501,ST01,SKU04596,CUST08781,1,335.55,335.55,In-Store,0.0,NaN
6999,2023-11-10,RCPT00003644,ST17,SKU04321,CUST06381,2,961.30,1922.60,In-Store,0.0,NaN
8031,2022-07-06,RCPT00004185,ST01,SKU04321,CUST05629,2,961.30,1922.60,In-Store,0.0,NaN
9251,2022-11-13,RCPT00004820,ST01,SKU04321,CUST09043,1,961.30,961.30,Online,0.0,NaN
9451,2022-01-03,RCPT00004929,ST19,SKU03119,CUST01169,1,37.56,37.56,In-Store,0.0,NaN


In [34]:
# Compare original and duplicate records
sales[sales.duplicated(keep=False)].sort_values(
    by=["receipt_id", "sku_id"]
).head(20)

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
270,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
272,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
314,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
315,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
544,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
545,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
2935,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
2936,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
5845,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN
5846,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN


In [35]:
# Remove duplicate records
sales = sales.drop_duplicates().copy()

In [36]:
# Verify that duplicate records have been removed
sales.duplicated().sum()

np.int64(0)

In [37]:
# Check the updated dataset dimensions
sales.shape

(9932614, 11)

In [38]:
sales.isnull().sum()

date                  0
receipt_id            0
store_id              0
sku_id                0
customer_id           0
quantity              0
unit_price            0
total_value           0
channel               0
discount_pct          0
promo_id        7845164
dtype: int64

In [39]:
# Check unique values before filling missing values
sales["promo_id"].value_counts(dropna=False)

promo_id
NaN         7845164
PROMO045     185574
PROMO010     181373
PROMO075     178522
PROMO022     136480
             ...   
PROMO025          5
PROMO029          5
PROMO053          4
PROMO027          3
PROMO012          3
Name: count, Length: 97, dtype: int64

In [40]:
# Replace missing promotion IDs with "No Promotion"
sales["promo_id"] = sales["promo_id"].fillna("No Promotion")

In [41]:
# Confirm there are no missing values remaining
sales["promo_id"].isnull().sum()

np.int64(0)

In [42]:
# Check quantity values less than or equal to zero
sales[sales["quantity"] <= 0]

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id


In [43]:
# Check unit prices less than or equal to zero
sales[sales["unit_price"] <= 0]

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id


In [44]:
# Check total values less than or equal to zero
sales[sales["total_value"] <= 0]

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id


In [45]:
# Check discount percentages outside the valid range (0–100)
sales[
    (sales["discount_pct"] < 0) |
    (sales["discount_pct"] > 100)
]

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id


In [46]:
# Display the cleaned dataset dimensions
sales.shape

(9932614, 11)

In [47]:
# Save the cleaned sales dataset
sales.to_csv(
    "../data/processed/sales_transactions_clean.csv",
    index=False
)

## Data Cleaning Summary

### Cleaning tasks completed

- Removed duplicate transaction records.
- Replaced missing promotion IDs with **"No Promotion"**.
- Converted the transaction date to datetime format.
- Validated quantity, unit price, total value, and discount percentage.
- Saved the cleaned Sales Transactions dataset to the processed data folder.

**Status:** ✅ Sales Transactions dataset successfully cleaned and ready for analysis.

# Inventory Snapshot Data Cleaning

## Objective

This section cleans the Inventory Snapshot dataset by identifying missing values, duplicate records, incorrect data types, and inconsistent inventory information to ensure accurate stock analysis.

In [48]:
# Preview the first 5 rows of the inventory dataset
inventory.head()

,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date
0,ST11,SKU02558,333,72,22,2025-06-23
1,ST21,SKU01031,236,67,14,2025-06-17
2,ST26,SKU02129,496,96,34,2025-07-10
3,ST19,SKU02907,109,34,13,2025-08-17
4,ST05,SKU01023,333,97,22,2025-07-12


In [49]:
# Check the number of rows and columns
inventory.shape

(26408, 6)

In [50]:
# Display dataset structure, data types, and non-null counts
inventory.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26408 entries, 0 to 26407
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   store_id           26408 non-null  object
 1   sku_id             26408 non-null  object
 2   stock_on_hand      26408 non-null  int64 
 3   reorder_point      26408 non-null  int64 
 4   safety_stock       26408 non-null  int64 
 5   last_restock_date  26408 non-null  object
dtypes: int64(3), object(3)
memory usage: 1.2+ MB


In [51]:
# Check for missing values in each column
inventory.isnull().sum()

store_id             0
sku_id               0
stock_on_hand        0
reorder_point        0
safety_stock         0
last_restock_date    0
dtype: int64

In [52]:
# Check for duplicate records
inventory.duplicated().sum()

np.int64(0)

In [53]:
# Generate descriptive statistics
inventory.describe(include="all")

,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date
count,26408,26408,26408.000000,26408.000000,26408.000000,26408
unique,30,4495,NaN,NaN,NaN,221
top,ST03,SKU04580,NaN,NaN,NaN,2025-08-31
freq,918,30,NaN,NaN,NaN,211
mean,NaN,NaN,166.195812,59.983301,20.951719,NaN
std,NaN,NaN,142.238780,23.389128,9.873781,NaN
min,NaN,NaN,0.000000,20.000000,4.000000,NaN
25%,NaN,NaN,40.000000,40.000000,13.000000,NaN
50%,NaN,NaN,140.000000,60.000000,20.000000,NaN
75%,NaN,NaN,264.000000,80.000000,28.000000,NaN


In [54]:
# Count unique values in each column
inventory.nunique()

store_id               30
sku_id               4495
stock_on_hand         595
reorder_point          81
safety_stock           47
last_restock_date     221
dtype: int64

In [55]:
# Display unique values for categorical columns
for col in inventory.select_dtypes(include="object").columns:
    print(f"\n{col}")
    print(inventory[col].value_counts(dropna=False))


store_id
store_id
ST03    918
ST29    914
ST30    913
ST21    907
ST06    898
ST10    894
ST15    893
ST16    889
ST17    885
ST12    883
ST25    882
ST05    882
ST04    882
ST01    882
ST09    882
ST14    880
ST28    880
ST11    879
ST23    877
ST26    877
ST08    874
ST22    870
ST18    869
ST24    868
ST07    861
ST19    858
ST27    857
ST02    854
ST20    852
ST13    848
Name: count, dtype: int64

sku_id
sku_id
SKU04580    30
SKU02780    30
SKU03058    30
SKU03773    30
SKU03251    30
            ..
SKU03055     1
SKU01978     1
SKU00556     1
SKU04451     1
SKU04275     1
Name: count, Length: 4495, dtype: int64

last_restock_date
last_restock_date
2025-08-31    211
2025-08-13    199
2025-08-24    197
2025-08-20    184
2025-08-09    184
             ... 
2025-09-08     50
2025-09-20     49
2025-09-25     49
2025-09-07     44
2025-09-13     40
Name: count, Length: 221, dtype: int64


In [56]:
# Convert last_restock_date to datetime
inventory["last_restock_date"] = pd.to_datetime(
    inventory["last_restock_date"],
    errors="coerce"
)

inventory["last_restock_date"].dtype

dtype('<M8[ns]')

In [57]:
# Check if any dates became missing after conversion
inventory["last_restock_date"].isnull().sum()

np.int64(0)

In [58]:
# Check for negative stock values
inventory[inventory["stock_on_hand"] < 0]

,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date


In [59]:
# Check for negative reorder points
inventory[inventory["reorder_point"] < 0]

,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date


In [60]:
# Check for negative safety stock values
inventory[inventory["safety_stock"] < 0]

,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date


In [61]:
# Check if safety stock exceeds reorder point
inventory[
    inventory["safety_stock"] >
    inventory["reorder_point"]
]

,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date


In [62]:
# Display cleaned dataset dimensions
inventory.shape

(26408, 6)

In [63]:
# Save the cleaned inventory dataset
inventory.to_csv(
    "../data/processed/inventory_snapshot_clean.csv",
    index=False
)

# Inventory Snapshot Data Cleaning Summary

## Summary

The Inventory Snapshot dataset was successfully cleaned and validated.

The following data quality checks were performed:

- Converted `last_restock_date` to datetime format.
- Verified that no missing values were introduced after conversion.
- Confirmed there were no duplicate records.
- Checked for negative stock quantities.
- Checked for negative reorder points.
- Checked for negative safety stock values.
- Verified that safety stock does not exceed reorder point.
- Saved the cleaned dataset for use in exploratory analysis and modeling.

The cleaned dataset is now consistent, complete, and ready for inventory analysis.

# SKU Master Data Cleaning

## Objective

This section cleans the SKU Master dataset by identifying missing values, duplicate records, incorrect data types, and inconsistent product information to ensure accurate product analysis and reliable downstream inventory and sales analysis.

In [65]:
# Preview the first 5 rows of the products dataset
sku.head()

,sku_id,sku_name,category,subcategory,unit_price,cost_price,brand
0,SKU00001,NutriPlus Cookware Large,Home & Kitchen,Cookware,813.41,619.77,NutriPlus
1,SKU00002,CrispKing Bread Family Pack,Dairy & Bakery,Bread,70.38,49.57,CrispKing
2,SKU00003,SoftTouch Notebooks 2L,Stationery & Office,Notebooks,151.28,83.67,SoftTouch
3,SKU00004,SunriseFoods Eggs Large,Dairy & Bakery,Eggs,233.64,156.32,SunriseFoods
4,SKU00005,SunriseFoods Pest Control Pack of 6,Home Care,Pest Control,138.50,83.46,SunriseFoods


In [66]:
# Check the number of rows and columns
sku.shape

(5000, 7)

In [67]:
# Display dataset structure, data types, and non-null counts
sku.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sku_id       5000 non-null   object 
 1   sku_name     5000 non-null   object 
 2   category     5000 non-null   object 
 3   subcategory  5000 non-null   object 
 4   unit_price   5000 non-null   float64
 5   cost_price   5000 non-null   float64
 6   brand        5000 non-null   object 
dtypes: float64(2), object(5)
memory usage: 273.6+ KB


In [68]:
# Check for missing values in each column
sku.isnull().sum()

sku_id         0
sku_name       0
category       0
subcategory    0
unit_price     0
cost_price     0
brand          0
dtype: int64

In [69]:
# Check for duplicate records
sku.duplicated().sum()

np.int64(0)

In [70]:
# Generate descriptive statistics
sku.describe(include="all")

,sku_id,sku_name,category,subcategory,unit_price,cost_price,brand
count,5000,5000,5000,5000,5000.000000,5000.000000,5000
unique,5000,4458,12,53,NaN,NaN,30
top,SKU00001,PremiumSelect Cheese Medium,Stationery & Office,Writing Instruments,NaN,NaN,CleanWave
freq,1,3,459,164,NaN,NaN,193
mean,NaN,NaN,NaN,NaN,619.191672,417.040736,NaN
std,NaN,NaN,NaN,NaN,708.009319,480.347009,NaN
min,NaN,NaN,NaN,NaN,24.410000,16.750000,NaN
25%,NaN,NaN,NaN,NaN,160.555000,108.825000,NaN
50%,NaN,NaN,NaN,NaN,318.745000,214.670000,NaN
75%,NaN,NaN,NaN,NaN,788.715000,530.027500,NaN


In [71]:
# Count unique values in each column
sku.nunique()

sku_id         5000
sku_name       4458
category         12
subcategory      53
unit_price     4837
cost_price     4791
brand            30
dtype: int64

In [72]:
# Display unique values for categorical columns
for col in sku.select_dtypes(include="object").columns:
    print(f"\n{col}")
    print(sku[col].value_counts(dropna=False))


sku_id
sku_id
SKU00001    1
SKU03331    1
SKU03338    1
SKU03337    1
SKU03336    1
           ..
SKU01667    1
SKU01666    1
SKU01665    1
SKU01664    1
SKU05000    1
Name: count, Length: 5000, dtype: int64

sku_name
sku_name
PremiumSelect Cheese Medium          3
MetroBrand Ready Meals 2kg           3
EverydayPlus Office Supplies 250g    3
TrueTaste Kids Wear Large            3
CityFresh Cheese Pack of 6           3
                                    ..
NatureBlend Batteries 500g           1
PremiumSelect Tea 500g               1
CrispKing Hair Care Pack of 12       1
PureLife Cheese Small                1
NovaFresh Ready Meals 500g           1
Name: count, Length: 4458, dtype: int64

category
category
Stationery & Office          459
Beverages                    457
Dairy & Bakery               428
Home Care                    425
Frozen Foods                 422
Snacks & Confectionery       414
Apparel & Footwear           410
Personal Care                410
Health & Wellness   

In [73]:
# Check unit prices less than or equal to zero
sku[sku["unit_price"] <= 0]

,sku_id,sku_name,category,subcategory,unit_price,cost_price,brand


In [74]:
# Check cost prices less than or equal to zero
sku[sku["cost_price"] <= 0]

,sku_id,sku_name,category,subcategory,unit_price,cost_price,brand


In [75]:
# Check products where selling price is lower than cost price
sku[sku["unit_price"] < sku["cost_price"]]

,sku_id,sku_name,category,subcategory,unit_price,cost_price,brand


In [76]:
# Display the cleaned dataset dimensions
sku.shape

(5000, 7)

In [77]:
# Save the cleaned SKU Master dataset
sku.to_csv(
    "../data/processed/sku_master_clean.csv",
    index=False
)

## SKU Master Data Cleaning Completed

### Summary

The SKU Master dataset was successfully cleaned and validated.

The following quality checks were performed:

- Verified dataset dimensions and structure.
- Confirmed correct data types.
- Checked for missing values.
- Verified there were no duplicate records.
- Reviewed descriptive statistics and unique values.
- Validated categorical fields (Category, Subcategory, Brand).
- Confirmed all unit prices are greater than zero.
- Confirmed all cost prices are greater than zero.
- Verified no products have a selling price lower than their cost price.

The cleaned dataset is now ready for exploratory data analysis, feature engineering, and integration with the Sales, Customer, Inventory, Store, and Promotions datasets.

# Store Master Data Cleaning

## Objective

This section cleans the Store Master dataset by identifying missing values, duplicate records, incorrect data types, and inconsistent store information to ensure accurate store-level analysis and reliable downstream reporting.

In [78]:
# Preview the first 5 rows of the store dataset
stores.head()

,store_id,store_name,city,store_type,opening_date
0,ST01,Quetta Convenience Store #01,Quetta,Convenience Store,2017-07-08
1,ST02,Islamabad Express Store #02,Islamabad,Express Store,2019-05-08
2,ST03,Sialkot Supermarket #03,Sialkot,Supermarket,2019-03-28
3,ST04,Multan Supermarket #04,Multan,Supermarket,2017-10-08
4,ST05,Karachi Supermarket #05,Karachi,Supermarket,2020-11-03


In [79]:
# Check the number of rows and columns
stores.shape

(30, 5)

In [80]:
# Display dataset structure, data types, and non-null counts
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   store_id      30 non-null     object
 1   store_name    30 non-null     object
 2   city          30 non-null     object
 3   store_type    30 non-null     object
 4   opening_date  30 non-null     object
dtypes: object(5)
memory usage: 1.3+ KB


In [81]:
# Check for missing values in each column
stores.isnull().sum()

store_id        0
store_name      0
city            0
store_type      0
opening_date    0
dtype: int64

In [82]:
# Check for duplicate records
stores.duplicated().sum()

np.int64(0)

In [83]:
# Generate descriptive statistics
stores.describe(include="all")

,store_id,store_name,city,store_type,opening_date
count,30,30,30,30,30
unique,30,30,12,4,30
top,ST01,Quetta Convenience Store #01,Karachi,Convenience Store,2017-07-08
freq,1,1,5,11,1


In [84]:
# Count unique values in each column
stores.nunique()

store_id        30
store_name      30
city            12
store_type       4
opening_date    30
dtype: int64

In [85]:
# Display unique values for categorical columns
for col in stores.select_dtypes(include="object").columns:
    print(f"\n{col}")
    print(stores[col].value_counts(dropna=False))


store_id
store_id
ST01    1
ST02    1
ST29    1
ST28    1
ST27    1
ST26    1
ST25    1
ST24    1
ST23    1
ST22    1
ST21    1
ST20    1
ST19    1
ST18    1
ST17    1
ST16    1
ST15    1
ST14    1
ST13    1
ST12    1
ST11    1
ST10    1
ST09    1
ST08    1
ST07    1
ST06    1
ST05    1
ST04    1
ST03    1
ST30    1
Name: count, dtype: int64

store_name
store_name
Quetta Convenience Store #01        1
Islamabad Express Store #02         1
Karachi Hypermarket #29             1
Karachi Supermarket #28             1
Rawalpindi Supermarket #27          1
Lahore Convenience Store #26        1
Quetta Convenience Store #25        1
Gujranwala Convenience Store #24    1
Sukkur Convenience Store #23        1
Islamabad Hypermarket #22           1
Peshawar Hypermarket #21            1
Multan Hypermarket #20              1
Sialkot Convenience Store #19       1
Karachi Supermarket #18             1
Faisalabad Supermarket #17          1
Lahore Convenience Store #16        1
Islamabad Convenience St

In [87]:
# Convert opening_date to datetime
stores["opening_date"] = pd.to_datetime(
    stores["opening_date"],
    errors="coerce"
)

In [88]:
# Check for invalid opening dates
stores[stores["opening_date"].isna()]

,store_id,store_name,city,store_type,opening_date


In [89]:
# Check if any store has an opening date in the future
stores[stores["opening_date"] > pd.Timestamp.today()]

,store_id,store_name,city,store_type,opening_date


In [90]:
# Display the cleaned dataset dimensions
stores.shape

(30, 5)

In [91]:
# Save the cleaned Store Master dataset
stores.to_csv(
    "../data/processed/store_master_clean.csv",
    index=False
)

## Store Master Data Cleaning Summary

### Cleaning Actions Performed

- Verified dataset dimensions.
- Checked data types and non-null counts.
- Confirmed there are no missing values.
- Verified there are no duplicate records.
- Reviewed descriptive statistics and unique categorical values.
- Converted the `opening_date` column to datetime format.
- Checked for invalid or future opening dates.
- Saved the cleaned dataset for downstream analysis.

### Outcome

The Store Master dataset passed all quality checks.

# Promotions Data Cleaning

## Objective

This section cleans the Promotions dataset by identifying missing values, duplicate records, incorrect data types, and inconsistent promotion information to ensure accurate promotional analysis and reliable downstream sales reporting.

In [92]:
# Preview the first 5 rows of the promotions dataset
promotions.head()

,promo_id,promo_name,start_date,end_date,discount_pct,promo_type,target_type,target_value
0,PROMO001,Spring Discount Days,2024-07-28,2024-08-02,15.1,BOGO,Brand,NovaFresh
1,PROMO002,Summer Savings Event,2023-03-09,2023-03-21,7.0,Bundle Offer,All,All
2,PROMO003,Payday Discount Days,2025-04-30,2025-05-26,10.2,BOGO,Brand,ZestyCo
3,PROMO004,Super Sale,2025-06-07,2025-06-17,15.9,Clearance,SKU,SKU01108
4,PROMO005,Festive Discount Days,2024-03-23,2024-04-08,46.0,Bundle Offer,All,All


In [93]:
# Check the number of rows and columns
promotions.shape

(100, 8)

In [94]:
# Display dataset structure, data types, and non-null counts
promotions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   promo_id      100 non-null    object 
 1   promo_name    100 non-null    object 
 2   start_date    100 non-null    object 
 3   end_date      100 non-null    object 
 4   discount_pct  100 non-null    float64
 5   promo_type    100 non-null    object 
 6   target_type   100 non-null    object 
 7   target_value  100 non-null    object 
dtypes: float64(1), object(7)
memory usage: 6.4+ KB


In [95]:
# Check for missing values in each column
promotions.isnull().sum()

promo_id        0
promo_name      0
start_date      0
end_date        0
discount_pct    0
promo_type      0
target_type     0
target_value    0
dtype: int64

In [96]:
# Check for duplicate records
promotions.duplicated().sum()

np.int64(0)

In [97]:
# Generate descriptive statistics
promotions.describe(include="all")

,promo_id,promo_name,start_date,end_date,discount_pct,promo_type,target_type,target_value
count,100,100,100,100,100.000000,100,100,100
unique,100,72,96,97,NaN,5,4,49
top,PROMO001,Payday Discount Days,2022-01-23,2023-04-18,NaN,Bundle Offer,Category,All
freq,1,5,2,2,NaN,23,38,19
mean,NaN,NaN,NaN,NaN,27.580000,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,14.134348,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,5.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,15.400000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,25.400000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,42.575000,NaN,NaN,NaN


In [98]:
# Count unique values in each column
promotions.nunique()

promo_id        100
promo_name       72
start_date       96
end_date         97
discount_pct     87
promo_type        5
target_type       4
target_value     49
dtype: int64

In [99]:
# Display unique values for categorical columns
for col in promotions.select_dtypes(include="object").columns:
    print(f"\n{col}")
    print(promotions[col].value_counts(dropna=False))


promo_id
promo_id
PROMO001    1
PROMO064    1
PROMO074    1
PROMO073    1
PROMO072    1
           ..
PROMO031    1
PROMO030    1
PROMO029    1
PROMO028    1
PROMO100    1
Name: count, Length: 100, dtype: int64

promo_name
promo_name
Payday Discount Days        5
Grand Blowout               4
Year-End Deal Days          4
Anniversary Offer           3
Mid-Season Discount Days    3
                           ..
Summer Savings Event        1
Spring Blowout              1
Year-End Offer              1
Clearance Offer             1
Festive Bonanza             1
Name: count, Length: 72, dtype: int64

start_date
start_date
2022-01-23    2
2024-09-26    2
2024-10-22    2
2022-01-30    2
2023-06-07    1
             ..
2022-11-16    1
2023-07-23    1
2023-10-27    1
2023-04-12    1
2022-10-25    1
Name: count, Length: 96, dtype: int64

end_date
end_date
2023-04-18    2
2023-04-12    2
2025-05-23    2
2024-08-02    1
2024-06-15    1
             ..
2025-08-13    1
2022-11-23    1
2023-08-08   

In [100]:
# Convert promotion dates to datetime
promotions["start_date"] = pd.to_datetime(
    promotions["start_date"],
    errors="coerce"
)

promotions["end_date"] = pd.to_datetime(
    promotions["end_date"],
    errors="coerce"
)

In [101]:
# Check invalid start dates
promotions[promotions["start_date"].isna()]

,promo_id,promo_name,start_date,end_date,discount_pct,promo_type,target_type,target_value


In [102]:
# Check invalid end dates
promotions[promotions["end_date"].isna()]

,promo_id,promo_name,start_date,end_date,discount_pct,promo_type,target_type,target_value


In [103]:
# Check promotions where end date is before start date
promotions[
    promotions["end_date"] < promotions["start_date"]
]

,promo_id,promo_name,start_date,end_date,discount_pct,promo_type,target_type,target_value


In [104]:
# Check discount percentages outside the valid range (0–100)
promotions[
    (promotions["discount_pct"] < 0) |
    (promotions["discount_pct"] > 100)
]

,promo_id,promo_name,start_date,end_date,discount_pct,promo_type,target_type,target_value


In [105]:
# Display the cleaned dataset dimensions
promotions.shape

(100, 8)

In [106]:
# Save the cleaned Promotions dataset
promotions.to_csv(
    "../data/processed/promotions_clean.csv",
    index=False
)

## Promotions Data Cleaning Summary

### Cleaning Actions Performed

- Verified dataset dimensions.
- Checked data types and non-null counts.
- Confirmed there are no missing values.
- Verified there are no duplicate records.
- Reviewed descriptive statistics and unique categorical values.
- Converted promotion start and end dates to datetime format.
- Checked for invalid promotion dates.
- Verified that promotion end dates occur after start dates.
- Confirmed discount percentages fall within the valid range (0–100).
- Saved the cleaned dataset for downstream analysis.

### Outcome

The Promotions dataset passed all quality checks. The cleaned dataset is complete, consistent, and ready for feature engineering, exploratory data analysis, and predictive modeling.

# SKU Inventory Flags Data Cleaning

## Objective

This section cleans the SKU Inventory Flags dataset by identifying missing values, duplicate records, incorrect data types, and inconsistent inventory flag information to ensure accurate inventory monitoring and reliable downstream analysis.

In [107]:
# Preview the first 5 rows of the SKU Inventory Flags dataset
flags.head()

,sku_id,flag,affected_stores,window_start,window_end,notes
0,SKU04321,STOCKOUT_RISK,ST22;ST26;ST23;ST15;ST12;ST28;ST01;ST27;ST08;S...,2025-11-06,2025-12-03,Top-selling SKU (by observed volume); injected...
1,SKU04596,STOCKOUT_RISK,ST12;ST09;ST21;ST23;ST07;ST24;ST01;ST28;ST14;S...,2025-10-25,2025-11-06,Top-selling SKU (by observed volume); injected...
2,SKU03727,STOCKOUT_RISK,ST19;ST14;ST28;ST29;ST02;ST07;ST17;ST08;ST05;ST16,2025-11-08,2025-11-29,Top-selling SKU (by observed volume); injected...
3,SKU04154,STOCKOUT_RISK,ST15;ST03;ST30;ST09;ST24;ST19;ST28;ST02;ST07;S...,2025-11-12,2025-11-26,Top-selling SKU (by observed volume); injected...
4,SKU00953,STOCKOUT_RISK,ST09;ST19;ST27;ST20;ST01;ST03;ST11;ST10;ST22;S...,2025-11-29,2025-12-25,Top-selling SKU (by observed volume); injected...


In [108]:
# Check the number of rows and columns
flags.shape

(600, 6)

In [109]:
# Display dataset structure, data types, and non-null counts
flags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   sku_id           600 non-null    object
 1   flag             600 non-null    object
 2   affected_stores  600 non-null    object
 3   window_start     200 non-null    object
 4   window_end       200 non-null    object
 5   notes            600 non-null    object
dtypes: object(6)
memory usage: 28.3+ KB


In [110]:
# Check for missing values in each column
flags.isnull().sum()

sku_id               0
flag                 0
affected_stores      0
window_start       400
window_end         400
notes                0
dtype: int64

In [111]:
# Check for duplicate records
flags.duplicated().sum()

np.int64(0)

In [112]:
# Generate descriptive statistics
flags.describe(include="all")

,sku_id,flag,affected_stores,window_start,window_end,notes
count,600,600,600,200,200,600
unique,600,2,600,60,62,2
top,SKU04321,SLOW_MOVER,ST22;ST26;ST23;ST15;ST12;ST28;ST01;ST27;ST08;S...,2025-10-29,2025-12-14,Bottom-selling SKU (by observed volume); injec...
freq,1,400,1,7,7,400


In [113]:
# Count unique values in each column
flags.nunique()

sku_id             600
flag                 2
affected_stores    600
window_start        60
window_end          62
notes                2
dtype: int64

In [114]:
# Display unique values for categorical columns
for col in flags.select_dtypes(include="object").columns:
    print(f"\n{col}")
    print(flags[col].value_counts(dropna=False))


sku_id
sku_id
SKU04321    1
SKU00890    1
SKU03619    1
SKU00251    1
SKU02681    1
           ..
SKU00012    1
SKU04953    1
SKU00502    1
SKU03805    1
SKU01863    1
Name: count, Length: 600, dtype: int64

flag
flag
SLOW_MOVER       400
STOCKOUT_RISK    200
Name: count, dtype: int64

affected_stores
affected_stores
ST22;ST26;ST23;ST15;ST12;ST28;ST01;ST27;ST08;ST21;ST24;ST07;ST30;ST20                                                                                    1
ST06;ST04;ST08;ST11;ST15;ST13;ST17;ST26;ST05;ST01;ST27;ST19;ST09;ST16;ST29;ST24;ST10;ST22                                                                1
ST28;ST22;ST29;ST20;ST27;ST24;ST11;ST17;ST16;ST14;ST21;ST09;ST07;ST01;ST30                                                                               1
ST04;ST29;ST12;ST16;ST06;ST03;ST05;ST21;ST30;ST26;ST11;ST18;ST20;ST17;ST25;ST19;ST23;ST01;ST10;ST15;ST28;ST09;ST27;ST07;ST02;ST13                        1
ST30;ST21;ST11;ST20;ST06;ST07;ST18;ST15;ST16;ST12;ST02;ST26;

In [115]:
# Count missing values in the monitoring window columns
flags[["window_start", "window_end"]].isnull().sum()

window_start    400
window_end      400
dtype: int64

In [116]:
# Check missing window dates by flag type
flags.loc[
    flags["window_start"].isna(),
    ["flag", "window_start", "window_end"]
]

,flag,window_start,window_end
200,SLOW_MOVER,NaN,NaN
201,SLOW_MOVER,NaN,NaN
202,SLOW_MOVER,NaN,NaN
203,SLOW_MOVER,NaN,NaN
204,SLOW_MOVER,NaN,NaN
...,...,...,...
595,SLOW_MOVER,NaN,NaN
596,SLOW_MOVER,NaN,NaN
597,SLOW_MOVER,NaN,NaN
598,SLOW_MOVER,NaN,NaN


In [120]:
flags["flag"].value_counts()

flag
SLOW_MOVER       400
STOCKOUT_RISK    200
Name: count, dtype: int64

In [121]:
# Check missing monitoring dates by flag type
flags.groupby("flag")[["window_start", "window_end"]].apply(
    lambda x: x.isnull().sum()
)

,window_start,window_end
flag,,
SLOW_MOVER,400,400
STOCKOUT_RISK,0,0


In [117]:
# Convert monitoring window columns to datetime
flags["window_start"] = pd.to_datetime(
    flags["window_start"],
    errors="coerce"
)

flags["window_end"] = pd.to_datetime(
    flags["window_end"],
    errors="coerce"
)

In [118]:
# Check records where the monitoring window ends before it starts
flags[
    (flags["window_start"].notna()) &
    (flags["window_end"].notna()) &
    (flags["window_end"] < flags["window_start"])
]

,sku_id,flag,affected_stores,window_start,window_end,notes


In [119]:
# Display the cleaned dataset dimensions
flags.shape

(600, 6)

In [122]:
flags.isnull().sum()

sku_id               0
flag                 0
affected_stores      0
window_start       400
window_end         400
notes                0
dtype: int64

In [123]:
flags[flags["flag"] == "SLOW_MOVER"][["flag", "window_start", "window_end"]].head()

,flag,window_start,window_end
200,SLOW_MOVER,NaT,NaT
201,SLOW_MOVER,NaT,NaT
202,SLOW_MOVER,NaT,NaT
203,SLOW_MOVER,NaT,NaT
204,SLOW_MOVER,NaT,NaT


In [124]:
flags[flags["flag"] == "STOCKOUT_RISK"][["flag", "window_start", "window_end"]].isnull().sum()

flag            0
window_start    0
window_end      0
dtype: int64

In [125]:
# Save cleaned SKU Inventory Flags dataset
flags.to_csv(
    "../data/processed/sku_inventory_flags_clean.csv",
    index=False
)

print("Cleaned SKU Inventory Flags dataset saved successfully.")

Cleaned SKU Inventory Flags dataset saved successfully.


## Cleaning Summary

The SKU Inventory Flags dataset was successfully validated and cleaned.

### Cleaning activities performed

- Verified dataset dimensions and structure.
- Checked for missing values across all columns.
- Confirmed there were no duplicate records.
- Converted `window_start` and `window_end` to datetime format.
- Validated monitoring windows to ensure end dates did not occur before start dates.
- Confirmed that missing monitoring dates only occurred for `SLOW_MOVER` records, where no monitoring window is required.
- Verified that all `affected_stores` and `notes` fields were populated.
- Confirmed there were no duplicate `sku_id` and `flag` combinations.

### Result

The dataset passed all quality checks and is ready for exploratory data analysis and downstream inventory risk monitoring.

# Data Cleaning Completed

## Overview

This notebook cleaned and validated all datasets used in the Project Foresight Demand & Inventory Intelligence project.

### Datasets cleaned

- Customer Master
- Sales Transactions
- Inventory Snapshot
- SKU Master
- Store Master
- Promotions
- SKU Inventory Flags

### Data quality checks performed

- Missing value detection
- Duplicate record detection
- Data type validation
- Range and consistency validation
- Date validation
- Categorical value inspection
- Business rule validation
- Saving cleaned datasets

### Output

All cleaned datasets have been exported to the `data/processed` directory and are ready for Exploratory Data Analysis (EDA), feature engineering, forecasting, and machine learning.